In [ ]:
# run_raster_parallel.py
from __future__ import annotations

import os
import multiprocessing as mp
import concurrent.futures as cf
from pathlib import Path

from general_utils import find_ephys_sessions
from raster_worker import process_session, OUTDIR  # import the top-level worker

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    _, _, sessions = find_ephys_sessions()
    print("Sessions:", sessions)
    if not sessions:
        return

    # Use spawn to avoid HDF5/NWB fork-safety issues
    ctx = mp.get_context("spawn")
    max_workers = min(len(sessions), os.cpu_count() or 1)
    print(f"Running in parallel with {max_workers} workers (spawn)")

    with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_session, s): s for s in sessions}
        for fut in cf.as_completed(futures):
            print(fut.result())

if __name__ == "__main__":
    # If something else already set the start method, this is fine.
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError:
        pass
    main()


In [ ]:
# move files to different subfolders
from pathlib import Path
import re
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

ROOT = Path("/root/capsule/scratch/raster_plot")
DRY_RUN = False

PNG_RE = re.compile(r"^(?P<prefix>.+)_unit_(?P<unit>\d+)\.png$", re.IGNORECASE)

def move_one(png_path: Path, dest_path: Path) -> None:
    if DRY_RUN:
        return
    png_path.rename(dest_path)

moved = 0
skipped = 0

max_workers = min(32, (os.cpu_count() or 8) * 2)
PRINT_EVERY = 500  # adjust

for session_dir in (p for p in ROOT.iterdir() if p.is_dir()):
    created = set()
    tasks = []

    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for png_path in session_dir.glob("*.png"):
            m = PNG_RE.match(png_path.name)
            if not m:
                skipped += 1
                continue

            prefix = m.group("prefix")
            dest_dir = session_dir / prefix

            if prefix not in created:
                if not DRY_RUN:
                    dest_dir.mkdir(parents=True, exist_ok=True)
                created.add(prefix)

            dest_path = dest_dir / png_path.name
            tasks.append(ex.submit(move_one, png_path, dest_path))

        total = len(tasks)
        done = 0

        for fut in as_completed(tasks):
            fut.result()
            moved += 1
            done += 1

            if done % PRINT_EVERY == 0 or done == total:
                print(f"[{session_dir.name}] {done}/{total} done | moved_total={moved} skipped={skipped}")

print("\nSummary")
print("Moved:", moved)
print("Skipped:", skipped)


In [3]:
# scatter_activity_latent_worker.py
from __future__ import annotations

import os
import multiprocessing as mp
import concurrent.futures as cf
from pathlib import Path

from general_utils import find_ephys_sessions
from scatter_activity_latent_worker import process_session, OUTDIR  # import the top-level worker

def main():
    OUTDIR.mkdir(parents=True, exist_ok=True)
    _, _, sessions = find_ephys_sessions()
    print("Sessions:", sessions)
    if not sessions:
        return

    # Use spawn to avoid HDF5/NWB fork-safety issues
    ctx = mp.get_context("spawn")
    max_workers = min(len(sessions), os.cpu_count() or 1)
    print(f"Running in parallel with {max_workers} workers (spawn)")

    with cf.ProcessPoolExecutor(max_workers=max_workers, mp_context=ctx) as ex:
        futures = {ex.submit(process_session, s): s for s in sessions}
        for fut in cf.as_completed(futures):
            print(fut.result())

if __name__ == "__main__":
    # If something else already set the start method, this is fine.
    try:
        mp.set_start_method("spawn", force=False)
    except RuntimeError:
        pass
    main()


Sessions: ['ecephys_753124_2024-12-10_17-24-56_sorted_2024-12-13_09-48-25', 'ecephys_753125_2024-10-09_10-50-19_sorted_2024-11-09_20-03-58', 'ecephys_753125_2024-10-10_14-41-23_sorted_2024-11-09_20-18-36', 'ecephys_753125_2024-10-14_15-37-15_sorted_2024-11-09_20-07-38', 'ecephys_753125_2024-10-15_16-16-22_sorted_2024-11-09_19-57-50', 'ecephys_753126_2024-10-10_17-51-24_sorted_2025-02-21_13-56-40', 'ecephys_753126_2024-10-11_13-14-24_sorted_2024-11-09_19-18-21', 'ecephys_753126_2024-10-15_12-20-35_sorted_2024-11-09_19-47-57', 'ecephys_764769_2024-12-11_18-21-49_sorted_2024-12-13_10-04-48', 'ecephys_764769_2024-12-12_16-05-00_sorted_2024-12-13_10-34-23', 'ecephys_764769_2024-12-13_15-41-07_sorted_2024-12-17_18-00-23', 'ecephys_764787_2024-12-10_13-42-03_sorted_2025-01-24_15-52-45', 'ecephys_764787_2024-12-11_15-01-15_sorted_2025-02-21_17-11-57', 'ecephys_764787_2024-12-12_11-54-14_sorted_2024-12-13_10-39-18', 'ecephys_764787_2024-12-13_18-27-42_sorted_2024-12-17_18-31-24', 'ecephys_76479